In [ ]:
from openai import OpenAI

#openai_client = OpenAI()
openai_client = OpenAI(
    api_key="",
    base_url="https://openrouter.ai/api/v1"
)

In [7]:
from pydantic import BaseModel

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

In [8]:
CalendarEvent.model_json_schema()

{'properties': {'name': {'title': 'Name', 'type': 'string'},
  'date': {'title': 'Date', 'type': 'string'},
  'participants': {'items': {'type': 'string'},
   'title': 'Participants',
   'type': 'array'}},
 'required': ['name', 'date', 'participants'],
 'title': 'CalendarEvent',
 'type': 'object'}

In [9]:
response = openai_client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role":"system", "content":"Extract the event information."},
        {
            "role": "user",
            "content": "Doomer and Chud are going to a mog event on Friday"
        }
    ],
    text_format=CalendarEvent
)

In [10]:
response.output[0].content[0].parsed

CalendarEvent(name='Mog Event', date='Friday', participants=['Doomer', 'Chud'])

In [11]:
response.output[0].content[0].text

'{"name":"Mog Event","date":"Friday","participants":["Doomer","Chud"]}'

In [12]:
event = response.output_parsed

In [13]:
event

CalendarEvent(name='Mog Event', date='Friday', participants=['Doomer', 'Chud'])

In [14]:
response = openai_client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
    text_format=CalendarEvent
)
print(response.output_text)

{"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}


In [15]:
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
)
print(response.output_text)

Event: Science Fair  
Participants: Alice and Bob  
Day: Friday


Event Information:
- Event: Science Fair
- Participants: Alice and Bob
- Day: Friday


Event: Science Fair  
Participants: Alice and Bob  
Day: Friday


### Structured RAG

In [17]:

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

Indexed 382 chunks from 95 documents


In [18]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

In [19]:

import json

instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

In [20]:
def llm(user_prompt, instructions=None, model="gpt-4o-mini"):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [21]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt, instructions)

In [22]:
answer = rag('how do I implement LLM as a Judge?')

In [23]:
def llm_structured(
        user_prompt,
        output_type,
        instructions=None,
        model="gpt-4o-mini",
    ):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    return response.output_parsed

In [24]:
response = llm_structured(
    instructions="Extract the event information.",
    user_prompt="Alice and Bob are going to a science fair on Friday.",
    output_type=CalendarEvent,
)

In [25]:
response

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

In [26]:
class RAGResponse(BaseModel):
    answer: str
    found_answer: bool

In [27]:
def rag_structured(query, output_type=RAGResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(
        instructions=instructions,
        user_prompt=prompt,
        output_type=output_type,
    )

In [28]:
answer = rag_structured('how do i do llm evals?')

print(answer.answer[:100])
print(answer.found_answer)

To conduct LLM evaluations, you can generate a synthetic dataset with questions, contexts, and respo
True


In [30]:
answer = rag_structured('how do I install kafka on windows?')

print(answer.answer[:100])
print(answer.found_answer)

The documentation does not contain information on how to install Kafka on Windows.
False


In [31]:
RAGResponse.model_json_schema()

{'properties': {'answer': {'title': 'Answer', 'type': 'string'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['answer', 'found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [32]:
from typing import Optional

class RAGResponse(BaseModel):
    answer: Optional[str] = None
    found_answer: bool

In [33]:
RAGResponse.model_json_schema()

{'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Answer'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [34]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


In [35]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


In [37]:

instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.

If you don't find the answer, set `answer` to None
"""

In [38]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


In [39]:

instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

In [40]:
class RAGResponse(BaseModel):
    """
    The response from the documentation RAG system

    If the answer to the question wasn't found in the database, `answer` is None
    """
    answer: Optional[str] = None
    found_answer: bool

In [41]:
RAGResponse.model_json_schema()

{'description': "The response from the documentation RAG system\n\nIf the answer to the question wasn't found in the database, `answer` is None",
 'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Answer'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [42]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


In [48]:
from pydantic import Field

class RAGResponse(BaseModel):
    """
    The response from the documentation RAG system
    """
    answer: Optional[str] = Field(None, description="Answer to the question or None if it's not found")
    found_answer: bool = Field(description="True if the answer is found, False otherwise")

In [49]:
RAGResponse.model_json_schema()

{'description': 'The response from the documentation RAG system',
 'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'description': "Answer to the question or None if it's not found",
   'title': 'Answer'},
  'found_answer': {'description': 'True if the answer is found, False otherwise',
   'title': 'Found Answer',
   'type': 'boolean'}},
 'required': ['found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [52]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


In [53]:
from typing import Literal

class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [54]:
RAGResponse.model_json_schema()

{'description': 'This model provides a structured answer with metadata about the response,\nincluding confidence, categorization, and follow-up suggestions.',
 'properties': {'answer': {'description': "The main answer to the user's question in markdown",
   'title': 'Answer',
   'type': 'string'},
  'found_answer': {'description': 'True if relevant information was found in the documentation',
   'title': 'Found Answer',
   'type': 'boolean'},
  'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
   'title': 'Confidence',
   'type': 'number'},
  'confidence_explanation': {'description': 'Explanation about the confidence level',
   'title': 'Confidence Explanation',
   'type': 'string'},
  'answer_type': {'description': 'The category of the answer',
   'enum': ['how-to',
    'explanation',
    'troubleshooting',
    'comparison',
    'reference'],
   'title': 'Answer Type',
   'type': 'string'},
  'followup_questions': {'description': 'S

In [55]:
answer = rag_structured('how do I evaluate llms', RAGResponse)


In [56]:
print(answer.answer[:100])
print(answer.confidence)
print(answer.confidence_explanation)
print(answer.answer_type)
print(answer.followup_questions)


To evaluate LLMs (Large Language Models), you can use a framework where multiple LLMs assess the sam
0.95
The information provided in the context gives clear steps and methods to evaluate LLMs based on using multiple LLMs as judges to review the same output, including specifics on setup, definitions, and report generation.
how-to
['What are the specific metrics to consider in LLM evaluation?', 'How can I customize the evaluation process based on my needs?', 'Can I use self-hosted LLMs for the evaluation?']


In [57]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)


In [58]:
print(answer.answer[:100])
print(answer.confidence)
print(answer.confidence_explanation)
print(answer.answer_type)
print(answer.followup_questions)


The documentation does not provide specific information on how to install Kafka on Windows.
0.0
There is no relevant information regarding Kafka installation in the provided documentation context.
reference
['Where can I find Kafka installation instructions for Windows?', 'What are the system requirements for running Kafka on Windows?', 'Are there any common issues when installing Kafka on Windows?']


In [59]:
from pydantic import model_validator


class AnswerNotFound(BaseModel):
    explanation: str


class AnswerResponse(BaseModel):
    """
    If answer is found, 'answer' is populated.
    If no answer is found, 'answer_not_found' is populated.
    Only one of the two fields can be set at a time. Never both or neither.
    """

    answer_not_found: Optional[AnswerNotFound] = None
    found_answer: bool
    answer: Optional[RAGResponse] = None

    @model_validator(mode="after")
    def check_consistency(self):
        if self.answer is not None and self.answer_not_found is not None:
            raise ValueError("Provide either 'answer' or 'answer_not_found', not both.")

        if self.answer is None and self.answer_not_found is None:
            raise ValueError("Provide either 'answer' or 'answer_not_found'.")

        return self

In [60]:
AnswerResponse.model_json_schema()

{'$defs': {'AnswerNotFound': {'properties': {'explanation': {'title': 'Explanation',
     'type': 'string'}},
   'required': ['explanation'],
   'title': 'AnswerNotFound',
   'type': 'object'},
  'RAGResponse': {'description': 'This model provides a structured answer with metadata about the response,\nincluding confidence, categorization, and follow-up suggestions.',
   'properties': {'answer': {'description': "The main answer to the user's question in markdown",
     'title': 'Answer',
     'type': 'string'},
    'found_answer': {'description': 'True if relevant information was found in the documentation',
     'title': 'Found Answer',
     'type': 'boolean'},
    'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
     'title': 'Confidence',
     'type': 'number'},
    'confidence_explanation': {'description': 'Explanation about the confidence level',
     'title': 'Confidence Explanation',
     'type': 'string'},
    'answer_type': 

In [61]:
answer = rag_structured('how do I install kafka on windows?', AnswerResponse)
answer

AnswerResponse(answer_not_found=AnswerNotFound(explanation='The provided context does not include information about installing Kafka on Windows.'), found_answer=False, answer=None)

In [62]:
answer = rag_structured('how do I run llm evals?', AnswerResponse)
answer

AnswerResponse(answer_not_found=None, found_answer=True, answer=RAGResponse(answer='To run LLM evaluations, follow these steps:\n\n1. **Install Evidently**:\n   ```bash\n   pip install evidently[llm]\n   ```\n\n2. **Import Required Modules**:\n   ```python\n   import pandas as pd\n   from evidently.future.datasets import Dataset\n   from evidently.future.datasets import DataDefinition\n   from evidently.future.report import Report\n   from evidently.future.presets import TextEvals\n   from evidently.ui.workspace.cloud import CloudWorkspace\n   ```\n\n3. **Connect to Evidently Cloud**:\n   ```python\n   ws = CloudWorkspace(token="YOUR_API_TOKEN", url="https://app.evidently.cloud")\n   ```\n\n4. **Create a Project**:\n   ```python\n   project = ws.create_project("Your Project Name", org_id="YOUR_ORG_ID")\n   project.description = "My project description"\n   project.save()\n   ```\n\n5. **Prepare Your Dataset**:\n   Create a toy dataset with questions and target responses:\n   ```python\

In [63]:
from pydantic import ValidationError

try:
    AnswerResponse()
except ValidationError as e:
    print("Validation error:")
    print(e)


Validation error:
1 validation error for AnswerResponse
found_answer
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
